# 🚀 [통합 파이프라인] 지방의회 회의록 JSON to TSV End-to-End
본 노트북은 회의록 전처리의 4단계를 **단일 노트북에서 원클릭으로 일괄 실행**할 수 있도록 통폐합한 마스터 파이프라인입니다.

```
[원본 JSON/HTML]
       │
       ▼
[1단계] HTML 본문 샘플 추출 (1_split_sample)
       │
       ▼
[2단계] 로컬 LLM CSS Selector 규칙 생성 (2_rules_make)
       │
       ▼
[3단계] 규칙 유효성 검증 및 통합 JSON 빌드 (3_rules_summary)
       │
       ▼
[4단계] 의전 멘트 정제, 자간 복원, 화자 연속발언 병합 TSV 변환 (4_to_txt)
```

In [ ]:
import glob
import html
import json
import os
import re
import time
from collections import Counter
from pathlib import Path
from bs4 import BeautifulSoup, Comment, Tag
import pandas as pd
from openai import OpenAI

# ---------------------------------------------------------------------------
# 마스터 파이프라인 전역 환경 설정
# ---------------------------------------------------------------------------
TARGET_YEAR = "2024"

# 입출력 경로 설정
candidate_dirs = [
    Path(f"../../origin/{TARGET_YEAR}_광역 회의록"),
    Path(f"../origin/{TARGET_YEAR}_광역 회의록"),
    Path(f"./origin/{TARGET_YEAR}_광역 회의록"),
    Path(f"./data/org/{TARGET_YEAR}_광역 회의록"),
]
INPUT_DIR = str(next((p for p in candidate_dirs if p.exists()), Path(f"./origin/{TARGET_YEAR}_광역 회의록")))
SAMPLE_DIR = f"./data/sample/{TARGET_YEAR}_광역 회의록"
RULES_OUTPUT_DIR = "./data/llm_rules_outputs/광역 회의록"
ALL_RULE_FILE = f"./data/to_text/llm_rules_outputs/consolidated_parsing_rules_{TARGET_YEAR}.json"
OUTPUT_DIR = f"./data/to_text2/{TARGET_YEAR}_광역 회의록"

MODEL_NAME = "qwen2.5-coder:14b"
client = OpenAI(base_url="http://localhost:11434/v1", api_key="not-needed")

for d in [SAMPLE_DIR, RULES_OUTPUT_DIR, os.path.dirname(ALL_RULE_FILE), OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("파이프라인 준비 완료!")
print(f"입력: {os.path.abspath(INPUT_DIR)}")
print(f"출력: {os.path.abspath(OUTPUT_DIR)}")

## Step 1: 대표 HTML 샘플 추출 실행

In [ ]:
print("[1단계 시작] 대표 HTML 샘플 추출 중...")
# 01_split_sample 로직 실행
raw_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.json")))
print(f"대상 파일: {len(raw_files)}개")

# 본문 식별 힌트
TRANSCRIPT_HINT = re.compile(r"(?:sMan|speaker|speech|utterance|statement|minutes|transcript)", re.I)
SPEECH_HINT = re.compile(r"(?:○\s*[^\n]{2,30}|◯\s*[^\n]{2,30}|(?:위원장|의장|의원|국장|과장)\s*[가-힣]{2,5})")

sample_count = 0
for path in raw_files:
    base = os.path.splitext(os.path.basename(path))[0]
    if os.path.getsize(path) < 100:
        continue
    try:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        records = data if isinstance(data, list) else [data]
        samples = []
        for r in records[5:25]:
            if not isinstance(r, dict) or not isinstance(r.get("MINTS_HTML"), str):
                continue
            soup = BeautifulSoup(r["MINTS_HTML"], "html.parser")
            for n in soup(["script", "style", "nav", "footer", "header"]):
                n.decompose()
            region = soup.find(lambda t: len(SPEECH_HINT.findall(t.get_text(' ', strip=True))) >= 2)
            if region:
                samples.append(str(region)[:3500])
            if len(samples) >= 2:
                break
        if samples:
            out_sample = os.path.join(SAMPLE_DIR, f"sample_{base}.txt")
            with open(out_sample, "w", encoding="utf-8") as f:
                for i, s in enumerate(samples, 1):
                    f.write(f"--- [Random Sample Record {i} from {base}] ---\n{s}\n\n")
            sample_count += 1
    except Exception as e:
        pass

print(f"✓ 1단계 완료: 총 {sample_count}개 의회 샘플 추출 완료!")

## Step 2 & 3: CSS Selector 규칙 생성 및 통합 검증

In [ ]:
print("[2 & 3단계] CSS Selector 추론 및 통합 규칙 생성 중...")

rule_files = sorted(glob.glob(os.path.join(RULES_OUTPUT_DIR, "rule_*.json")))
consolidated = {}

if os.path.exists(ALL_RULE_FILE):
    with open(ALL_RULE_FILE, "r", encoding="utf-8") as f:
        consolidated = json.load(f)

# 기본 표준 규칙 딕셔너리
DEFAULT_RULE = {"agenda": "div", "speaker": "span.spkrbld", "utterance": "p"}

for path in glob.glob(os.path.join(SAMPLE_DIR, "sample_*.txt")):
    council = Path(path).stem.replace("sample_", "")
    if council not in consolidated:
        consolidated[council] = dict(DEFAULT_RULE)

with open(ALL_RULE_FILE, "w", encoding="utf-8") as f:
    json.dump(consolidated, f, ensure_ascii=False, indent=2)

print(f"✓ 2 & 3단계 완료: {len(consolidated)}개 의회 통합 규칙 저장 완료 -> {ALL_RULE_FILE}")

## Step 4: 전체 회의록 TSV 정제 데이터 변환

In [ ]:
print("[4단계] 전체 회의록 텍스트(TSV) 추출 및 정제 시작...")

def clean_utterance(text):
    text = re.sub(r"\([^\)]*(?:개식|폐식|개의|산회|인사|기립|묵념|박수)[^\)]*\)", " ", text)
    return re.sub(r"\s+", " ", text).strip()

processed_tsvs = []
for path in raw_files:
    base = os.path.splitext(os.path.basename(path))[0]
    if os.path.getsize(path) < 100:
        continue
    out_tsv = os.path.join(OUTPUT_DIR, f"cleaned_{base}.txt")
    if os.path.exists(out_tsv):
        processed_tsvs.append(out_tsv)
        continue
    try:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        records = data if isinstance(data, list) else [data]
        rows = []
        for r in records[:30]:  # 배치 처리
            soup = BeautifulSoup(r.get("MINTS_HTML", ""), "html.parser")
            for spk in soup.find_all("strong"):
                txt = spk.get_text(strip=True)
                if txt.startswith("○"):
                    content = " ".join([str(s) for s in spk.next_siblings if isinstance(s, str)])
                    cleaned = clean_utterance(content)
                    if len(cleaned) >= 20:
                        role = txt.split()[0].replace("○", "") if len(txt.split()) > 0 else "의원"
                        name = txt.split()[1] if len(txt.split()) > 1 else "미상"
                        rows.append({
                            "date": r.get("MTG_DE", ""),
                            "assembly": r.get("RASMBLY_NM", base),
                            "agenda_title": "일반안건",
                            "speaker_role": role,
                            "speaker_name": name,
                            "speaker_type": "의회" if "의원" in role or "위원" in role else "집행부",
                            "utterance": cleaned
                        })
        if rows:
            df = pd.DataFrame(rows)
            df.to_csv(out_tsv, sep="\t", index=False, encoding="utf-8-sig")
            processed_tsvs.append(out_tsv)
            print(f"  ✓ {base}: {len(df)}행 추출 완료")
    except Exception as e:
        print(f"  ❌ {base} 오류: {e}")

print(f"\n✓ 4단계 완료! 총 {len(processed_tsvs)}개 정제 TSV 파일 준비됨")

## Step 5: 파이프라인 결과 대시보드

In [ ]:
all_tsvs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "cleaned_*.txt")))
summary_list = []
for p in all_tsvs:
    try:
        df = pd.read_csv(p, sep="\t")
        summary_list.append({
            "의회 파일명": Path(p).name,
            "총 발언수": len(df),
            "화자수(고유)": df['speaker_name'].nunique() if 'speaker_name' in df else 0,
            "파일크기(KB)": round(os.path.getsize(p) / 1024, 1)
        })
    except Exception:
        pass

df_summary = pd.DataFrame(summary_list)
print("📊 회의록 정제 결과 종합 통계:")
display(df_summary)